#India trade overview

A. India Trade Overview
Questions:
Total exports vs imports over time
Trade balance (profit/loss)
KPIs:
Total Export Value
Total Import Value
Trade Balance = Export − Import
Power BI visuals:
Line chart (year-wise trend)
KPI cards


In [0]:
from pyspark.sql.functions import *

In [0]:
df=spark.read.table("raw.raw_schema.raw_trade")

In [0]:

trade_summary=df.groupBy("trade_year", "trade_type").agg(sum("trade_value_usd").alias("total_trade")).orderBy(desc("trade_year"))
trade_summary.show()

+----------+----------+------------+
|trade_year|trade_type| total_trade|
+----------+----------+------------+
|      2025|    Import|754345639154|
|      2025|    Export|445070597791|
|      2024|    Export|442829716737|
|      2024|    Import|716509694497|
|      2023|    Import|671151030600|
|      2023|    Export|449282479377|
|      2022|    Import|720195942477|
|      2022|    Export|453259802672|
|      2021|    Import|573173917825|
|      2021|    Export|395471826573|
|      2020|    Export|276467086082|
|      2020|    Import|373283175986|
|      2019|    Export|324267556306|
|      2019|    Import|485949143637|
|      2018|    Import|514555504553|
|      2018|    Export|324843545617|
|      2017|    Export|299152029381|
|      2017|    Import|449795403358|
|      2016|    Export|264424354105|
|      2016|    Import|361496571404|
+----------+----------+------------+
only showing top 20 rows


In [0]:
trade_pivot=trade_summary.groupBy("trade_year").pivot("trade_type",["Export","Import"]).sum("total_trade")
trade_pivot.show()

+----------+------------+------------+
|trade_year|      Export|      Import|
+----------+------------+------------+
|      2025|445070597791|754345639154|
|      2024|442829716737|716509694497|
|      2023|449282479377|671151030600|
|      2022|453259802672|720195942477|
|      2021|395471826573|573173917825|
|      2020|276467086082|373283175986|
|      2019|324267556306|485949143637|
|      2018|324843545617|514555504553|
|      2017|299152029381|449795403358|
|      2016|264424354105|361496571404|
|      2015|267791985807|393831433491|
|      2014|322480472804|462920226764|
|      2013|314815169622|465401773262|
|      2012|296828100007|489693903291|
|      2011|302904844736|464461771301|
|      2010|225030730777|350232876254|
+----------+------------+------------+



In [0]:
trade_final=trade_pivot.withColumn("trade_balance", col("Export")-col("Import")).select("trade_year","trade_balance")
trade_final.show()

+----------+-------------+
|trade_year|trade_balance|
+----------+-------------+
|      2025|-309275041363|
|      2024|-273679977760|
|      2023|-221868551223|
|      2022|-266936139805|
|      2021|-177702091252|
|      2020| -96816089904|
|      2019|-161681587331|
|      2018|-189711958936|
|      2017|-150643373977|
|      2016| -97072217299|
|      2015|-126039447684|
|      2014|-140439753960|
|      2013|-150586603640|
|      2012|-192865803284|
|      2011|-161556926565|
|      2010|-125202145477|
+----------+-------------+



+---------------------------+
|(sum(Export) + sum(Import))|
+---------------------------+
|             13651918306248|
+---------------------------+



In [0]:
g.show()

+----------+------------+------------+-------------+
|trade_year|      Export|      Import|trade_balance|
+----------+------------+------------+-------------+
|      2025|445070597791|754345639154|-309275041363|
|      2024|442829716737|716509694497|-273679977760|
|      2023|449282479377|671151030600|-221868551223|
|      2022|453259802672|720195942477|-266936139805|
|      2021|395471826573|573173917825|-177702091252|
|      2020|276467086082|373283175986| -96816089904|
|      2019|324267556306|485949143637|-161681587331|
|      2018|324843545617|514555504553|-189711958936|
|      2017|299152029381|449795403358|-150643373977|
|      2016|264424354105|361496571404| -97072217299|
|      2015|267791985807|393831433491|-126039447684|
|      2014|322480472804|462920226764|-140439753960|
|      2013|314815169622|465401773262|-150586603640|
|      2012|296828100007|489693903291|-192865803284|
|      2011|302904844736|464461771301|-161556926565|
|      2010|225030730777|350232876254|-1252021

B. Top Commodities Analysis



In [0]:
# top 10 export commodity
export_df=df.where("trade_type='Export'")
top_export=export_df.groupBy("commodity").agg(sum("trade_value_usd").alias("total_trade")).orderBy(col("total_trade").desc()).limit(10)
top_export.show()


+--------------------+------------+
|           commodity| total_trade|
+--------------------+------------+
|  PETROLEUM PRODUCTS|849915775172|
|PEARL, PRECS, SEM...|376098121863|
|DRUG FORMULATIONS...|237425301375|
|GOLD AND OTH PREC...|189704235859|
|      IRON AND STEEL|164413862022|
|RMG COTTON INCL A...|140577332587|
|PRODUCTS OF IRON ...|122401029999|
| TELECOM INSTRUMENTS|118843085960|
|ELECTRIC MACHINER...|118632116301|
|  MOTOR VEHICLE/CARS|115697688900|
+--------------------+------------+



In [0]:
# top 10 import commodity_df=df.where("trade_type='Import'")
import_df=df.where("trade_type='Import'")
top_import=import_df.groupBy("commodity").agg(sum("trade_value_usd").alias("total_trade")).orderBy(col("total_trade").desc()).limit(10)
top_import.show()

+--------------------+-------------+
|           commodity|  total_trade|
+--------------------+-------------+
|    PETROLEUM: CRUDE|1826781992809|
|                GOLD| 643916585722|
|  PETROLEUM PRODUCTS| 430612844069|
|PEARL, PRECS, SEM...| 401384531062|
|COAL,COKE AND BRI...| 370260322902|
|ELECTRONICS COMPO...| 256091613407|
| TELECOM INSTRUMENTS| 252760986622|
|      IRON AND STEEL| 202242975428|
|   ORGANIC CHEMICALS| 200257406850|
|INDL. MACHNRY FOR...| 192898422129|
+--------------------+-------------+



C. Top Partner Countries

In [0]:
#export destination
export_df.groupBy("country").agg(sum("trade_value_usd").alias("total_trade")).orderBy(col("total_trade").desc()).limit(10).show()

+-------------+------------+
|      country| total_trade|
+-------------+------------+
|        U S A|856232232038|
|  U ARAB EMTS|504236471372|
|   CHINA P RP|246548153880|
|    SINGAPORE|180261394893|
|    HONG KONG|180072594669|
|   NETHERLAND|176499588104|
|          U K|158623462495|
|      GERMANY|135995467300|
|BANGLADESH PR|133892751493|
|   SAUDI ARAB|131101071282|
+-------------+------------+



In [0]:
#import source
import_df.groupBy("country").agg(sum("trade_value_usd").alias("total_trade")).orderBy(col("total_trade").desc()).limit(10).show()

+-----------+-------------+
|    country|  total_trade|
+-----------+-------------+
| CHINA P RP|1174437632807|
|U ARAB EMTS| 578288084116|
|      U S A| 508817164609|
| SAUDI ARAB| 450106243390|
|SWITZERLAND| 345321205027|
|       IRAQ| 327152163044|
|     RUSSIA| 284623350915|
|  INDONESIA| 266077106404|
|   KOREA RP| 248799462642|
|    GERMANY| 231570135609|
+-----------+-------------+



D. Port-Level Analysis



In [0]:
port_trade=df.groupBy("port").agg(sum("trade_value_usd").alias("total_trade")).orderBy(col("total_trade").desc()).show()

+--------------------+-------------+
|                port|  total_trade|
+--------------------+-------------+
|     NHAVA SHEVA SEA|1706918046213|
|         OTHER PORTS|1431358330870|
|           DELHI AIR| 794587392919|
|         CHENNAI SEA| 775698559651|
|              MUNDRA| 560423870232|
|          MUMBAI AIR| 513885471735|
|SEZ Jamnagar (Rel...| 470185886724|
|          MUMBAI SEA| 403995176088|
|               SIKKA| 393305748000|
|             VADINAR| 364485655990|
|         KOLKATA SEA| 356584714393|
|         CHENNAI AIR| 345417564269|
|         DPCC MUMBAI| 339547940098|
|         PARADIP SEA| 338092536130|
|   VISAKHAPATNAM SEA| 311645736948|
|   BANGALORE AIRPORT| 287051086179|
|          KANDLA SEA| 279978560754|
|          COCHIN SEA| 217543746907|
|   NEWMANGALORE  SEA| 210959753365|
|       TUTICORIN SEA| 198538385867|
+--------------------+-------------+
only showing top 20 rows


In [0]:
#import vs export by port
port_trade=df.groupBy("port", "trade_type").agg(sum("trade_value_usd").alias("total_trade")).orderBy("port")
port_trade.show()

+--------------------+----------+-----------+
|                port|trade_type|total_trade|
+--------------------+----------+-----------+
| KIP SEZ NORTH BA...|    Export|     342125|
| KIP SEZ NORTH BA...|    Import|  173793592|
|AC JAIPUR (GEMS A...|    Import| 1508989929|
|AC JAIPUR (GEMS A...|    Export|  561035557|
|ACC AMRITSAR (RAJ...|    Import| 4935936770|
|ACC AMRITSAR (RAJ...|    Export|  161692827|
|ACC CALICUT, KARIPUR|    Export|  266979679|
|ACC CALICUT, KARIPUR|    Import|    3715985|
|      ACC COIMBATORE|    Export|  357252721|
|      ACC COIMBATORE|    Import| 1345031791|
|     ACC GOA,DABOLIM|    Export|  216452186|
|     ACC GOA,DABOLIM|    Import|  391450794|
|  ACC JANORI (NASIK)|    Export|  261940810|
|  ACC JANORI (NASIK)|    Import|   21220232|
|       ACC MOPA, GOA|    Import|   19844526|
|       ACC MOPA, GOA|    Export|   39321552|
|           ADANI ICD|    Import|  165541591|
|           ADANI ICD|    Export|  332615748|
|ADANI ICD KILA RA...|    Export| 

In [0]:
port_trade.groupBy("port").pivot("trade_type",["Export","Import"]).sum("total_trade").show()

+--------------------+-----------+-----------+
|                port|     Export|     Import|
+--------------------+-----------+-----------+
| KIP SEZ NORTH BA...|     342125|  173793592|
|AC JAIPUR (GEMS A...|  561035557| 1508989929|
|ACC AMRITSAR (RAJ...|  161692827| 4935936770|
|ACC CALICUT, KARIPUR|  266979679|    3715985|
|      ACC COIMBATORE|  357252721| 1345031791|
|     ACC GOA,DABOLIM|  216452186|  391450794|
|  ACC JANORI (NASIK)|  261940810|   21220232|
|       ACC MOPA, GOA|   39321552|   19844526|
|           ADANI ICD|  332615748|  165541591|
|ADANI ICD KILA RA...|  759745231|  894046742|
|ADANI POWER JHARK...| 1734710166|   22099571|
|ADANI POWER LTD  ...|  746873898| 1114516445|
|ADARSH PRIME LTD ...|   17174918|  161600255|
|AEQUES SEZ, BHANA...|     245994|       NULL|
|AFS KAPASHERA, BI...|     317777|       NULL|
|            AGARTALA|   25457049|  389452257|
|AHMEDABAD AIR CAR...|35485560151|78558554980|
|AHMEDABAD APPAREL...|  192010601|   58296837|
|    AIR CARG

🎯 How Power BI Uses This
Table	Visual
india_trade_overview	Line chart + KPI
top_export_commodities	Bar / Tree
top_export_countries	Map
port_trade_analysis	Stacked chart

In [0]:
    df.show()

+--------------------+-----------+-----------------+----+---+---------------+----------+----------+
|           commodity|    country|             port|unit|qty|trade_value_usd|trade_type|trade_year|
+--------------------+-----------+-----------------+----+---+---------------+----------+----------+
|WOLLEN YARN,FABRI...|      JAPAN|      KOLKATA AIR|NULL|  0|              0|    Import|      2010|
|WOLLEN YARN,FABRI...|      KENYA|      DELHI (ICD)|NULL|  0|           7781|    Import|      2010|
|WOLLEN YARN,FABRI...|KOREA DP RP|      DELHI (ICD)|NULL|  0|           7226|    Import|      2010|
|WOLLEN YARN,FABRI...|   KOREA RP|      DELHI (ICD)|NULL|  0|         164989|    Import|      2010|
|WOLLEN YARN,FABRI...|   KOREA RP|      CHENNAI SEA|NULL|  0|         107990|    Import|      2010|
|WOLLEN YARN,FABRI...|   KOREA RP|  NHAVA SHEVA SEA|NULL|  0|         105817|    Import|      2010|
|WOLLEN YARN,FABRI...|   KOREA RP|    ICD BANGALORE|NULL|  0|          12534|    Import|      2010|
